Following the data and simulation files processing, here we:

- Merge processed files by production conditions.
- Tag global events by type of particle (electron or alpha-like) and detector region.
- Store final HDF5 merged file for subsequent analysis.

In [1]:
import sys
sys.path.append('/lhome/ific/c/ccortesp/Analysis/')

from libs import crudo

import os
import pandas as pd
import numpy as np

%matplotlib inline
%load_ext autoreload
%autoreload 2

# Configuration

In [ ]:
# -----------------------------------------
# 1. DIRECTORIES, PATHS, KEYS AND FILENAMES
# -----------------------------------------
# FILENAME TAG
VERSION_TAG = 'p2_zemrude'      # or 'p2_final'

# DIRECTORIES, PATHS & FILES
PROCESSED_DIR = '/lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Backgrounds/h5/runs/'

RUNS_INFO_PATH = os.path.join('/lhome/ific/c/ccortesp/Analysis/NEXT-100/Backgrounds/utilities/runs_information.csv')

SUMMARY_FILENAME = 'summary_' + VERSION_TAG +'_processed.csv'     # Choose your name
SUMMARY_PATH = os.path.join('/lhome/ific/c/ccortesp/Analysis/NEXT-100/Backgrounds/txt/summaries/', SUMMARY_FILENAME)

# # KEYS
# DORO_KEY = 'DST/Events'
# SOPH_KEY = 'RECO/Events'
# EVT_KEY  = 'PROCESSED/Events'

# # COLUMNS TO USE
# DORO_COLUMNS = ['event', 'time', 'nS1', 'nS2', 'S1h', 'S1e', 'S2e', 'DT', 'X', 'Y', 'Z']
# SOPH_COLUMNS = ['event', 'time', 'npeak', 'X', 'Y', 'Z', 'Q', 'E']
# FINAL_SOPH_COLUMNS = ['event', 'time', 'npeak', 'X', 'Y', 'DT', 'Z', 'E_hit_pe']

# # CUTFLOWS
# CUT_NAMES = ['Reconstructed', 'Z_Positive', 'S1_Cut', 'Clean_Hits']

# --------------------------------
# 2. ALPHA/ELECTRON DISCRIMINATION
# --------------------------------
SIZE_THRESHOLD = 2e3          # in [# of hits]
S1_ENERGY_THRESHOLD = 900     # in [PE]

# -------------------
# 3. DETECTOR REGIONS
# -------------------
# Geometric boundaries for event classification.
Z_LOW = 40          # in [mm]
Z_UP  = 1147        # in [mm]
R_UP  = 451.65      # in [mm]

## Runs & Summary Information

In [ ]:
# Runs information
RUNS_INFO_DF = pd.read_csv(RUNS_INFO_PATH)
RUNS_INFO_DF.columns = RUNS_INFO_DF.columns.str.strip()
RUNS_INFO_DF

,run_number,duration,OK,LOST,period,condition
0,15062,84783,69564,1339,1,castle_open
1,15063,79120,65052,1241,1,castle_open
2,15076,69316,56775,1080,1,castle_open
3,15288,87256,30201,8397,1,castle_pclosed_RAS
4,15289,82152,28180,7884,1,castle_pclosed_RAS
...,...,...,...,...,...,...
106,15733,86919,30475,10027,2,castle_closed_RAS
107,15734,85790,29837,9598,2,castle_closed_RAS
108,15735,87451,30547,9958,2,castle_closed_RAS
109,15736,93376,32622,10506,2,castle_closed_RAS


In [ ]:
# Summary of the processed runs
SUMMARY_DF = pd.read_csv(SUMMARY_PATH)
SUMMARY_DF.drop(columns=['Unnamed: 0'], inplace=True)
SUMMARY_DF.columns = SUMMARY_DF.columns.str.strip()
SUMMARY_DF.sort_values(by='Run_ID', inplace=True)
SUMMARY_DF

,Run_ID,Duration,Date_CV,Date_Err,OK,LOST,Reconstructed,Z_Positive,S1_Cut,Clean_Events
1,15730,87572,1.758680e+09,174.7509,30329,9960,30017,22188,18848,18842
2,15731,86561,1.758767e+09,174.5081,30155,9790,29765,21928,18538,18528
3,15732,85314,1.758854e+09,174.3647,29641,9508,29319,21600,18337,18330
4,15733,86919,1.758939e+09,175.0424,30475,10027,30144,22120,18663,18657
5,15734,85790,1.759026e+09,176.3429,29837,9598,29513,21813,18275,18263
6,15735,87451,1.759113e+09,175.5404,30547,9958,30193,22198,18810,18801
7,15736,93376,1.759204e+09,181.1020,32622,10506,32243,23718,20280,20272
0,15737,52343,1.759276e+09,134.7178,18365,5844,18192,13440,11449,11446


# Merge by Production Condition

- For data, the production are differenciated by _data period_ and _detector condition_.
- For MC, ...

In [18]:
# ----- Configuration ----- #
# Data
DATA_PERIOD = 2                 # Options: 1, 2
DETECTOR_CONDITION = 'castle_closed_RAS'       # Options: None, 'castle_open', 'castle_closed', 'castle_closed_RAS', 'castle_pclosed', 'castle_pclosed_RAS'

# Simulations

In [17]:
# Select runs to use according to the notebook configuration
if DATA_PERIOD is not None:
    runs_to_analyze = RUNS_INFO_DF.loc[RUNS_INFO_DF['period'] == DATA_PERIOD, 'run_number'].values
    if DETECTOR_CONDITION is not None:
        runs_to_analyze = RUNS_INFO_DF.loc[(RUNS_INFO_DF['period'] == DATA_PERIOD) & (RUNS_INFO_DF['condition'] == DETECTOR_CONDITION), 'run_number'].values

# Selection
print(f"\nSelected {len(runs_to_analyze)} runs for merge:")
print(runs_to_analyze)


Selected 56 runs for merge:
[15625 15626 15627 15632 15633 15634 15635 15636 15637 15639 15640 15642
 15643 15644 15645 15647 15648 15649 15650 15655 15656 15657 15658 15659
 15669 15670 15671 15672 15673 15675 15676 15681 15682 15687 15688 15689
 15693 15694 15695 15696 15697 15698 15699 15700 15701 15709 15724 15729
 15730 15731 15732 15733 15734 15735 15736 15737]


In [ ]:
total_corr_time = 0
# total_ok = 0
# total_lost = 0
total_processed_events = 0
all_processed_df = []

for run_id in runs_to_analyze:

    print(f"--- Merging Run {run_id} ---")
    if run_id not in SUMMARY_DF['Run_ID'].values:
        print(f"  → Run {run_id} not found in summary file. Skipping...")
        continue

    # --- Run Information --- #
    # Extract run information from the summary dataframe
    run_duration = SUMMARY_DF.loc[SUMMARY_DF['Run_ID'] == run_id, 'Duration'].values[0]
    run_OK   = SUMMARY_DF.loc[SUMMARY_DF['Run_ID'] == run_id, 'OK'].values[0]
    run_LOST = SUMMARY_DF.loc[SUMMARY_DF['Run_ID'] == run_id, 'LOST'].values[0]
    # Calculate DAQ efficiency and corrected time
    DAQe_CV, DAQe_error = crudo.ff.efficiency(run_OK, run_LOST)
    run_corr_time    = run_duration * DAQe_CV
    total_corr_time += run_corr_time
    # Accumulate processed events
    processed_events = SUMMARY_DF.loc[SUMMARY_DF['Run_ID'] == run_id, 'Clean_Events'].values[0]
    total_processed_events += processed_events

    # Add run_number to dataframe for the global_event_id
    run_file = os.path.join(PROCESSED_DIR, f'processed_run_{run_id}_{VERSION_TAG}.h5')
    run_df = pd.read_hdf(run_file, key='Events')
    run_df['run_number'] = run_id
    all_processed_df.append(run_df)

# --- Print Summary --- #
print(f"\nFor period {DATA_PERIOD} with condition '{DETECTOR_CONDITION}':\n  Corrected Time = {total_corr_time:.4f} s")
# Concatenate all dataframes
MERGED_DF = pd.concat(all_processed_df, ignore_index=True)
print(f"Dataframe merged successfully:\n  Total processed events: {total_processed_events}")

--- Merging Run 15625 ---
  --> Run 15625 not found in summary file. Skipping...
--- Merging Run 15626 ---
  --> Run 15626 not found in summary file. Skipping...
--- Merging Run 15627 ---
  --> Run 15627 not found in summary file. Skipping...
--- Merging Run 15632 ---
  --> Run 15632 not found in summary file. Skipping...
--- Merging Run 15633 ---
  --> Run 15633 not found in summary file. Skipping...
--- Merging Run 15634 ---
  --> Run 15634 not found in summary file. Skipping...
--- Merging Run 15635 ---
  --> Run 15635 not found in summary file. Skipping...
--- Merging Run 15636 ---
  --> Run 15636 not found in summary file. Skipping...
--- Merging Run 15637 ---
  --> Run 15637 not found in summary file. Skipping...
--- Merging Run 15639 ---
  --> Run 15639 not found in summary file. Skipping...
--- Merging Run 15640 ---
  --> Run 15640 not found in summary file. Skipping...
--- Merging Run 15642 ---
  --> Run 15642 not found in summary file. Skipping...
--- Merging Run 15643 ---
  

### Compute Global Event ID

In [21]:
# An original event is defined as a row in dataframe where at least one of the columns 
# ('event', 'run_number') differs from the corresponding row below it (using `shift`).
event_OG = (MERGED_DF[['event', 'run_number']] != MERGED_DF[['event', 'run_number']].shift())

# If any column in event_OG is True, it means the row corresponds to the start of a new original event block.
new_event_block = event_OG.any(axis=1)

# Use `cumsum()` on the boolean mask to create a unique identifier for each contiguous block of hits 
# that belong to the same original event.
unique_block_id = new_event_block.cumsum()

# Assign a unique global event ID to each block of original events.
# The `factorize` function generates a unique integer code for each unique block ID.
MERGED_DF['global_event'] = pd.factorize(unique_block_id)[0]
print(f"{MERGED_DF['global_event'].nunique()} unique global events identified.")

143139 unique global events identified.


In [22]:
MERGED_DF

,event,npeak,E_peak_pe,X_bary,Y_bary,Z_bary,Z_min,Z_max,R_max,E_evt_pe,...,main_S1w,main_S1h,main_S1t,main_S2e,main_S2w,main_S2h,main_S2t,main_S2q,run_number,global_event
0,232,30,1.434496e+05,-428.793838,144.922488,618.913986,609.526467,629.397247,506.334036,1.434496e+05,...,875.0,181.284637,693875.0,1.138413e+05,172.075,8571.013672,1409486.625,44701.003906,15730,0
1,484,28,2.822459e+05,230.889682,358.727952,342.160938,333.884702,353.694283,521.781603,2.822459e+05,...,725.0,149.672852,1012775.0,2.305865e+05,127.375,23215.917969,1408486.750,62015.050781,15730,1
2,1191,24,9.911036e+05,-88.293267,-45.818346,1183.873765,1165.907550,1203.204187,529.419957,9.911036e+05,...,375.0,240.813843,49700.0,9.954204e+05,224.000,55368.937500,1418489.000,234418.406250,15730,2
3,1401,23,9.407084e+05,228.457502,-269.079337,1185.848513,1161.282827,1205.388421,546.045608,9.407084e+05,...,650.0,221.696396,47200.0,8.200094e+05,123.000,44657.898438,1418490.000,194460.625000,15730,3
4,2402,25,1.563125e+05,-425.611641,-160.477972,148.028625,139.745940,155.553274,497.539532,1.563125e+05,...,350.0,107.181808,1233625.0,1.248611e+05,172.675,17615.271484,1404489.500,43775.574219,15730,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156682,1789053,29,9.653148e+05,83.784219,253.578880,1184.813041,1163.695637,1207.938333,506.672815,9.653148e+05,...,650.0,251.366013,49450.0,9.181305e+05,119.700,50818.015625,1419487.875,199074.328125,15737,143134
156683,1789270,26,9.864601e+05,143.002184,156.799348,1184.806588,1164.829003,1209.152468,500.123641,9.864601e+05,...,900.0,233.034393,49025.0,9.645358e+05,118.200,52852.453125,1419486.375,207234.468750,15737,143135
156684,1790145,32,8.675041e+05,-25.302845,110.439333,40.506878,19.056382,63.463104,520.044009,8.675041e+05,...,NaN,NaN,NaN,8.981901e+05,138.800,50272.562500,1417486.750,203066.671875,15737,143136
156685,1790215,33,1.058043e+06,-15.709436,-138.562164,898.007263,878.418313,919.020439,499.423574,1.058043e+06,...,675.0,222.353699,377450.0,1.044777e+06,110.275,67887.304688,1415487.375,211491.031250,15737,143137


# Tagging Events

### By Particle

In [ ]:
particle_tagged_MERGED_DF = crudo.dm.tag_particles(MERGED_DF, size_threshold=SIZE_THRESHOLD, s1_energy_threshold=S1_ENERGY_THRESHOLD, event_column='global_event')
particle_tagged_MERGED_DF

### By Detector Region

In [ ]:
region_tagged_MERGED_DF = crudo.dm.tag_event_by_detector_region(particle_tagged_MERGED_DF, z_cut_low=Z_LOW, z_cut_high=Z_UP, r_cut_high=R_UP, event_column='global_event')
region_tagged_MERGED_DF

# Output

In [27]:
# H5 output filename
MERGED_FILENAME = f'merged_tagged_runs_{DETECTOR_CONDITION}_{VERSION_TAG}.h5'
MERGED_PATH = os.path.join('/lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Backgrounds/h5/', MERGED_FILENAME)
print(f"\nSaving merged dataframe to: {MERGED_PATH}")


Saving merged dataframe to: /lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Backgrounds/h5/merged_tagged_runs_castle_closed_RAS_p2_zemrude.h5


In [28]:
region_tagged_MERGED_DF.to_hdf(MERGED_PATH, key='Events', mode='w', format='table')
print('Done!')

Done!
